In [1]:
import pandas as pd
import numpy as np

crop_df = pd.read_csv("../dataset/raw/crop_production.csv")

rainfall_df = pd.read_csv("../dataset/raw/rainfall.csv", sep=";")

population_df = pd.read_csv("../dataset/raw/population.csv")

In [2]:
population_df["Population"]= (population_df["Population"].astype(str).str.replace(",", "", regex=False))
population_df["Population"] = pd.to_numeric(population_df["Population"], errors= "coerce")


In [3]:
print(population_df["Population"].dtype)

int64


In [4]:
crop_df.columns = crop_df.columns.str.strip()
population_df.columns = population_df.columns.str.strip()
rainfall_df.columns = rainfall_df.columns.str.strip()

In [5]:
print(crop_df.columns.tolist())
print(population_df.columns.tolist())
print(rainfall_df.columns.tolist())

['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield']
['Ranking', 'District', 'State', 'Population', 'Growth', 'Sex-Ratio', 'Literacy']
['state', 'district', 'month', '1st', '2nd', '3rd', '4th', '5th', '6th', '7th', '8th', '9th', '10th', '11th', '12th', '13th', '14th', '15th', '16th', '17th', '18th', '19th', '20th', '21st', '22nd', '23rd', '24th', '25th', '26th', '27th', '28th', '29th', '30th', '31st']


In [6]:
crop_df = crop_df.rename(columns={"State": "state", "District": "district", "Crop": "crop", "Season": "season", "Area": "area", "Production": "production" })
population_df = population_df.rename(columns={"State": "state","District": "district", "Population": "population"})

In [7]:
crop_df["state"] = crop_df["state"].astype(str).str.strip()
crop_df["district"] = crop_df["district"].astype(str).str.lower().str.strip()

population_df["state"] = population_df["state"].astype(str).str.strip()
population_df["district"] = population_df["district"].astype(str).str.lower().str.strip()

rainfall_df["state"] = rainfall_df["state"].astype(str).str.strip()
rainfall_df["district"] = rainfall_df["district"].astype(str).str.lower().str.strip()

In [8]:
print(crop_df.columns.tolist())
print(population_df.columns.tolist())
print(rainfall_df.columns.tolist())

['state', 'district', 'crop', 'Crop_Year', 'season', 'area', 'production', 'Yield']
['Ranking', 'district', 'state', 'population', 'Growth', 'Sex-Ratio', 'Literacy']
['state', 'district', 'month', '1st', '2nd', '3rd', '4th', '5th', '6th', '7th', '8th', '9th', '10th', '11th', '12th', '13th', '14th', '15th', '16th', '17th', '18th', '19th', '20th', '21st', '22nd', '23rd', '24th', '25th', '26th', '27th', '28th', '29th', '30th', '31st']


In [9]:
rainfall_cols = [
    col
    for col in rainfall_df.columns
    if col not in ["state", "district", "month"]
]

rainfall_df["monthly_rainfall"] = (
    rainfall_df[rainfall_cols]
    .mean(axis=1)
)

rainfall_df = (
    rainfall_df
    .groupby(
        ["state", "district"],
        as_index=False
    )["monthly_rainfall"]
    .mean()
)

In [10]:
print(rainfall_df.shape)

print(
    rainfall_df[
        ["state", "district"]
    ].duplicated().sum()
)

(863, 3)
0


In [11]:
final_df = crop_df.merge(
    population_df[
        ["state", "district", "population"]
    ],
    on=["state", "district"],
    how="left"
)

final_df = final_df.merge(
    rainfall_df,
    on=["state", "district"],
    how="left"
)

In [12]:
print(final_df.shape)

print(final_df.columns.tolist())

final_df.head()

(345336, 10)
['state', 'district', 'crop', 'Crop_Year', 'season', 'area', 'production', 'Yield', 'population', 'monthly_rainfall']


,state,district,crop,Crop_Year,season,area,production,Yield,population,monthly_rainfall
0,Andaman and Nicobar Island,nicobars,Arecanut,2007,Kharif,2439.6,3415.0,1.40,NaN,NaN
1,Andaman and Nicobar Island,nicobars,Arecanut,2007,Rabi,1626.4,2277.0,1.40,NaN,NaN
2,Andaman and Nicobar Island,nicobars,Arecanut,2008,Autumn,4147.0,3060.0,0.74,NaN,NaN
3,Andaman and Nicobar Island,nicobars,Arecanut,2008,Summer,4147.0,2660.0,0.64,NaN,NaN
4,Andaman and Nicobar Island,nicobars,Arecanut,2009,Autumn,4153.0,3120.0,0.75,NaN,NaN


In [13]:
print(final_df.shape)

(345336, 10)


In [14]:
final_df["production"] = (
    final_df["production"]
    .fillna(final_df["production"].median())
)

final_df["population"] = (
    final_df["population"]
    .fillna(final_df["population"].median())
)

final_df["monthly_rainfall"] = (
    final_df["monthly_rainfall"]
    .fillna(final_df["monthly_rainfall"].median())
)

In [15]:
final_df["demand"] = (
    final_df["production"] * 0.50
    + final_df["population"] * 0.002
    + final_df["monthly_rainfall"] * 5
).round(0)

In [16]:
ratio = final_df["production"] / final_df["demand"]

final_df["risk_level"] = np.where(
    ratio >= 1.2,
    "Low Risk",
    np.where(
        ratio >= 0.8,
        "Moderate Risk",
        "High Risk"
    )
)

In [17]:
print(final_df["risk_level"].value_counts())

risk_level
High Risk        240369
Low Risk          79050
Moderate Risk     25917
Name: count, dtype: int64


In [18]:
final_df.to_csv(
    "../dataset/processed/final_dataset.csv",
    index=False
)

final_df.to_json(
    "../frontend/public/final_dataset.json",
    orient="records"
)

print("Dataset Saved")


Dataset Saved
